# [D3~D5] 모듈 ① & ② 최종 평가 및 리포팅

이 노트북은 D1~D2 단계 이후, `.py` 모듈로 작성된 D3~D5 파이프라인(Stacking, LambdaMART, XAI, 공정성 평가)을 Colab에서 손쉽게 순서대로 실행하기 위해 만들어졌습니다.

## 1. 최신 코드 불러오기 및 환경 설정

In [ ]:
# 1. 작업 디렉토리 이동
%cd /content/thisabled-ai

# 2. 최신 코드 불러오기
!git pull origin main

# 3. 필요 패키지 설치
!pip install shap fairlearn lightgbm sentence_transformers -q


fatal: not a git repository (or any of the parent directories): .git
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 70.8 MB/s eta 0:00:00:00:0100:01


## 2. 모듈 ① (위험도 분류) 평가

아래 경로(`/content/drive/MyDrive/...`)는 이전 노트북(`02_train_module1.ipynb`)에서 성공적으로 저장한 KcELECTRA 베이스라인 모델의 실제 경로입니다.

In [2]:
MODEL_DIR = "/content/drive/MyDrive/ThisAbled/checkpoints/module1_baseline_20260531_1427/checkpoint-7400"
STACKER_PATH = "models/checkpoints/module1_stacking.pkl"

print("=== 2.1 Fairlearn 공정성 지표 산출 ===")
!python scripts/evaluate_fairness.py --model-dir {MODEL_DIR} --stacker-path {STACKER_PATH}

print("\n=== 2.2 합성 Hold-out 셋 (긴급 3) 성능 산출 ===")
!python scripts/evaluate_synthetic.py --model-dir {MODEL_DIR}

=== 2.1 Fairlearn 공정성 지표 산출 ===
python3: can't open file '/content/scripts/evaluate_fairness.py': [Errno 2] No such file or directory

=== 2.2 합성 Hold-out 셋 (긴급 3) 성능 산출 ===
python3: can't open file '/content/scripts/evaluate_synthetic.py': [Errno 2] No such file or directory


## 3. SHAP XAI 해석 (HTML 리포트 생성)
오분류된 텍스트에서 모델이 어떤 단어에 집중하여 틀렸는지를 보여주는 시각화 결과물(`.html`)을 생성합니다.

In [3]:
!python scripts/evaluate_shap.py --model-dir {MODEL_DIR}

# 생성된 HTML 파일을 확인하려면 Colab 좌측 폴더 탭에서:
# reports/validation_reports/module1/shap/shap_misclassifications_top10.html 다운로드

python3: can't open file '/content/scripts/evaluate_shap.py': [Errno 2] No such file or directory


## 4. 모듈 ② (사용자 호환성 매칭) NDCG 평가
LambdaMART 모델의 랭킹 성능(NDCG@1, 5, 10)을 평가합니다.

In [4]:
!python scripts/evaluate_module2.py

python3: can't open file '/content/scripts/evaluate_module2.py': [Errno 2] No such file or directory


In [1]:
import os
from pathlib import Path
print(f"CWD: {os.getcwd()}")
print(f"repo exists: {Path('/content/thisabled-ai').exists()}")
if Path('/content/thisabled-ai').exists():
    !cd /content/thisabled-ai && git log --oneline -3


CWD: /content
repo exists: False


In [3]:
%cd /content/thisabled-ai
!git pull
!git log --oneline -3


/content/thisabled-ai
Already up to date.
38b2200 (HEAD -> main, origin/main, origin/HEAD) EXP-1, EXP-2: 긴급(3) Recall 목표 + 모듈 ② leakage 차단
f2c6534 docs: update final report with actual evaluation metrics
f88ea83 fix(eval): fix pipeline parsing in evaluate_shap.py


In [2]:
%cd /content
import shutil
from pathlib import Path
if Path("/content/thisabled-ai").exists():
    shutil.rmtree("/content/thisabled-ai")

import getpass
token = getpass.getpass("Token: ")
!git clone https://{token}@github.com/threeGuineas/thisabled-ai.git /content/thisabled-ai
%cd /content/thisabled-ai
!git log --oneline -3


/content
Cloning into '/content/thisabled-ai'...
remote: Enumerating objects: 198, done.
remote: Counting objects: 100% (198/198), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 198 (delta 75), reused 161 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (198/198), 5.59 MiB | 25.80 MiB/s, done.
Resolving deltas: 100% (75/75), done.
/content/thisabled-ai
38b2200 (HEAD -> main, origin/main, origin/HEAD) EXP-1, EXP-2: 긴급(3) Recall 목표 + 모듈 ② leakage 차단
f2c6534 docs: update final report with actual evaluation metrics
f88ea83 fix(eval): fix pipeline parsing in evaluate_shap.py


In [4]:
%cd /content/thisabled-ai
!python scripts/download_seed_datasets.py
!python scripts/build_processed_dataset.py
!python scripts/build_final_dataset.py --synth-repeat 8


/content/thisabled-ai
target: /content/thisabled-ai/data/raw
[unsmile/train] downloading
  -> https://raw.githubusercontent.com/smilegate-ai/korean_unsmile_dataset/main/unsmile_train_v1.0.tsv
[unsmile/train] ok (1,848,905 bytes, 15,005 rows)
[unsmile/valid] downloading
  -> https://raw.githubusercontent.com/smilegate-ai/korean_unsmile_dataset/main/unsmile_valid_v1.0.tsv
[unsmile/valid] ok (455,315 bytes, 3,737 rows)
[kold] downloading (LFS)
  -> https://media.githubusercontent.com/media/boychaboy/KOLD/main/data/kold_v1.json
[kold] ok (32,791,434 bytes, sha256 verified)
done.
=== Splits saved ===
[train] n=47,336  labels={0: 19836, 1: 9460, 2: 18040}  sources={'kold': 32291, 'unsmile': 15045}  -> data/processed/train.parquet
[val] n=5,917  labels={0: 2479, 1: 1183, 2: 2255}  sources={'kold': 4111, 'unsmile': 1806}  -> data/processed/val.parquet
[test] n=5,918  labels={0: 2480, 1: 1183, 2: 2255}  sources={'kold': 4027, 'unsmile': 1891}  -> data/processed/test.parquet
=== 통합 최종 데이터셋 빌드 (s

In [10]:
from pathlib import Path
candidates = [
    "/content/drive/MyDrive/ThisAbled/synthetic/emergency_v1",
    "/content/drive/MyDrive/ThisAbled/synthetic/emergency",
    "/content/drive/MyDrive/ThisAbled/data/synthetic/emergency",
]
for p in candidates:
    pp = Path(p)
    print(f"{p}: {'✓ 존재' if pp.exists() else '✗ 없음'}")
    if pp.exists():
        for jsonl in pp.rglob("*.jsonl"):
            n = len(jsonl.read_text().strip().splitlines())
            print(f"  {jsonl.relative_to(pp)}: {n}건")


/content/drive/MyDrive/ThisAbled/synthetic/emergency_v1: ✗ 없음
/content/drive/MyDrive/ThisAbled/synthetic/emergency: ✗ 없음
/content/drive/MyDrive/ThisAbled/data/synthetic/emergency: ✗ 없음


In [7]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


Mounted at /content/drive


In [20]:
!pip install -q google-genai

import getpass, os
os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY 붙여넣기: ")
# 키 자체는 출력 안 됨 — 단 길이만 확인
print(f"키 길이: {len(os.environ['GEMINI_API_KEY'])}자")


키 길이: 53자


In [14]:
%cd /content/thisabled-ai
!python scripts/synthesize_emergency.py --dry-run


/content/thisabled-ai
카테고리: ['3a', '3b', '3c', '3d', 'boundary']
모델: gemini-2.5-flash (무료 tier, 15 RPM)
batch_size: 20, 호출 간격: 4.5초
출력 경로: data/synthetic/emergency/

[3a/train] 추가 생성 필요: 200건
  → 10 LLM 호출, 예상 시간 ~45초
[3a/val] 추가 생성 필요: 30건
  → 2 LLM 호출, 예상 시간 ~9초
[3a/test] 추가 생성 필요: 50건
  → 3 LLM 호출, 예상 시간 ~14초
[3b/train] 추가 생성 필요: 150건
  → 8 LLM 호출, 예상 시간 ~36초
[3b/val] 추가 생성 필요: 25건
  → 2 LLM 호출, 예상 시간 ~9초
[3b/test] 추가 생성 필요: 40건
  → 2 LLM 호출, 예상 시간 ~9초
[3c/train] 추가 생성 필요: 150건
  → 8 LLM 호출, 예상 시간 ~36초
[3c/val] 추가 생성 필요: 25건
  → 2 LLM 호출, 예상 시간 ~9초
[3c/test] 추가 생성 필요: 40건
  → 2 LLM 호출, 예상 시간 ~9초
[3d/train] 추가 생성 필요: 100건
  → 5 LLM 호출, 예상 시간 ~22초
[3d/val] 추가 생성 필요: 20건
  → 1 LLM 호출, 예상 시간 ~4초
[3d/test] 추가 생성 필요: 30건
  → 2 LLM 호출, 예상 시간 ~9초
[boundary/train] 추가 생성 필요: 200건
  → 10 LLM 호출, 예상 시간 ~45초
[boundary/val] 추가 생성 필요: 50건
  → 3 LLM 호출, 예상 시간 ~14초
[boundary/test] 추가 생성 필요: 60건
  → 3 LLM 호출, 예상 시간 ~14초

=== 요약 ===
{
  "3a": {
    "train": 0,
    "val": 0,
    "test": 0
  },
  "3b": 

In [22]:
import os
os.environ["GEMINI_API_KEY"] = input("새 GEMINI_API_KEY: ").strip()
print(f"키 길이: {len(os.environ['GEMINI_API_KEY'])}자")


키 길이: 53자


In [24]:
!python scripts/synthesize_emergency.py


카테고리: ['3a', '3b', '3c', '3d', 'boundary']
모델: gemini-2.0-flash-lite (무료 tier, 15 RPM)
batch_size: 20, 호출 간격: 4.5초
출력 경로: data/synthetic/emergency/

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jsonschema/_format.py", line 304, in <module>
    import rfc3987
ModuleNotFoundError: No module named 'rfc3987'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py", line 80, in <module>
    import jsonschema
  File "/usr/local/lib/python3.12/dist-packages/jsonschema/_

In [21]:
!python scripts/synthesize_emergency.py


카테고리: ['3a', '3b', '3c', '3d', 'boundary']
모델: gemini-2.0-flash-lite (무료 tier, 15 RPM)
batch_size: 20, 호출 간격: 4.5초
출력 경로: data/synthetic/emergency/

[3a/train] 추가 생성 필요: 180건
  ⚠ 호출 실패 (1/5): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and , 30초 후 재시도
object address  : 0x7edc95971c60
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


In [18]:
%cd /content/thisabled-ai
!sed -i 's/MODEL = "gemini-2.0-flash-lite"/MODEL = "gemini-2.0-flash"/' scripts/synthesize_emergency.py
!grep "^MODEL" scripts/synthesize_emergency.py


/content/thisabled-ai
MODEL = "gemini-2.0-flash"  # 무료 tier — 2.0-flash 쿼터 소진으로 전환


In [17]:
!python scripts/synthesize_emergency.py


카테고리: ['3a', '3b', '3c', '3d', 'boundary']
모델: gemini-2.0-flash (무료 tier, 15 RPM)
batch_size: 20, 호출 간격: 4.5초
출력 경로: data/synthetic/emergency/

[3a/train] 추가 생성 필요: 180건
  ⚠ 호출 실패 (1/5): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and , 30초 후 재시도
object address  : 0x7fd6684e5ba0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


In [19]:
!sed -i 's/MODEL = "gemini-2.0-flash"/MODEL = "gemini-2.0-flash-lite"/' scripts/synthesize_emergency.py
!python scripts/synthesize_emergency.py


카테고리: ['3a', '3b', '3c', '3d', 'boundary']
모델: gemini-2.0-flash-lite (무료 tier, 15 RPM)
batch_size: 20, 호출 간격: 4.5초
출력 경로: data/synthetic/emergency/

[3a/train] 추가 생성 필요: 180건
  ⚠ 호출 실패 (1/5): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and , 30초 후 재시도
  ⚠ 호출 실패 (2/5): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and , 30초 후 재시도
  ⚠ 호출 실패 (3/5): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and , 30초 후 재시도
object address  : 0x7ea3bb47dae0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


In [9]:
import sys, os
os.chdir("/content/thisabled-ai")
if "/content/thisabled-ai" not in sys.path:
    sys.path.insert(0, "/content/thisabled-ai")
for mod in list(sys.modules.keys()):
    if mod.startswith("src.") or mod == "src":
        del sys.modules[mod]

from src.training.trainer import train_module1
result_final = train_module1("configs/module1_kcelectra_final.yaml", project_root=".")

import json
print("\n=== EXP-1 학습 완료 ===")
print(json.dumps(result_final["eval_metrics"], indent=2, ensure_ascii=False))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/504 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/450k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base-v2022
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the che

model.safetensors:   0%|          | 0.00/511M [00:00<?, ?B/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [8]:
from pathlib import Path
candidates = [
    "/content/drive/MyDrive/ThisAbled/synthetic/emergency_v1",
    "/content/drive/MyDrive/ThisAbled/synthetic/emergency",
    "/content/drive/MyDrive/ThisAbled/data/synthetic/emergency",
]
for p in candidates:
    pp = Path(p)
    print(f"{p}: {'✓ 존재' if pp.exists() else '✗ 없음'}")
    if pp.exists():
        for jsonl in pp.rglob("*.jsonl"):
            n = len(jsonl.read_text().strip().splitlines())
            print(f"  {jsonl.relative_to(pp)}: {n}건")


/content/drive/MyDrive/ThisAbled/synthetic/emergency_v1: ✗ 없음
/content/drive/MyDrive/ThisAbled/synthetic/emergency: ✗ 없음
/content/drive/MyDrive/ThisAbled/data/synthetic/emergency: ✗ 없음


In [6]:
import shutil
from pathlib import Path

# 위에서 ✓ 뜬 경로로 수정
src = Path("/content/drive/MyDrive/ThisAbled/synthetic/emergency_v1")
dst = Path("data/synthetic/emergency")
dst.parent.mkdir(parents=True, exist_ok=True)
if dst.exists(): shutil.rmtree(dst)
shutil.copytree(src, dst)
print(f"✓ 복원: {dst}")
!find data/synthetic/emergency -name "*.jsonl" -exec wc -l {} \;


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ThisAbled/synthetic/emergency_v1'

In [1]:
# 5) baseline_test (있으면 로드)
import json
m1_dir = Path("models/checkpoints/module1")
for j in m1_dir.glob("baseline_*.json"):
    baseline_json = json.loads(j.read_text())
    baseline_test = baseline_json["test_metrics"]
    print(f"\n✓ baseline_test 로드: macro_f1={baseline_test['macro_f1']:.4f}")
    break

NameError: name 'Path' is not defined

In [2]:
from pathlib import Path
print("repo:", "✓" if Path("/content/thisabled-ai").exists() else "✗")
print("data/processed:", "✓" if Path("/content/thisabled-ai/data/processed/test.parquet").exists() else "✗")
print("합성:", "✓" if Path("/content/thisabled-ai/data/synthetic/emergency").exists() else "✗")


repo: ✗
data/processed: ✗
합성: ✗


In [3]:
%cd /content
import os, sys, shutil
from pathlib import Path

# 1) 재클론
import getpass
token = input("GitHub Token (입력 후 Enter): ").strip()
if Path("/content/thisabled-ai").exists():
    shutil.rmtree("/content/thisabled-ai")
!git clone https://{token}@github.com/threeGuineas/thisabled-ai.git /content/thisabled-ai

# 2) cwd · sys.path 정리
os.chdir("/content/thisabled-ai")
sys.path.insert(0, "/content/thisabled-ai")

# 3) 데이터 파이프라인 (시드 다운로드 → split → 합성 → 통합)
!python scripts/download_seed_datasets.py
!python scripts/build_processed_dataset.py
!python scripts/synthesize_emergency_template.py   # LLM 없음, 5초
!python scripts/build_final_dataset.py --synth-repeat 8

# 4) Drive 마운트 + 베이스라인 복원 (있으면)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

baseline_src = Path("/content/drive/MyDrive/ThisAbled/checkpoints/module1_baseline_20260531_1302")
baseline_dst = Path("models/checkpoints/module1")
if baseline_src.exists():
    if baseline_dst.exists():
        shutil.rmtree(baseline_dst)
    shutil.copytree(baseline_src, baseline_dst)
    import json
    j = baseline_dst / "baseline_20260531_1302.json"
    if j.exists():
        baseline_json = json.loads(j.read_text())
        baseline_test = baseline_json["test_metrics"]
        print(f"\n✓ baseline_test: macro_f1={baseline_test['macro_f1']:.4f}")
else:
    print("\n⚠ Drive에 베이스라인 없음 — 비교 시 None 처리")
    baseline_test = None

# 5) 상태 확인
print("\n=== 최종 상태 ===")
!ls data/processed/
print()
!ls data/synthetic/emergency/


Mounted at /content/drive

✓ baseline_test: macro_f1=0.7464

=== 최종 상태 ===
synthetic_holdout.parquet  test.parquet  train.parquet	val.parquet

3a  3b	3c  3d	boundary


In [4]:
import pandas as pd
df = pd.read_parquet("data/processed/train.parquet")
dist = df["label"].value_counts().sort_index()
total = len(df)
print(f"train 총 {total:,}건")
print(f"라벨 분포: {dict(dist)}")
print(f"긴급(3) 비율: {dist.get(3, 0)/total*100:.2f}%  ← 5% 이상이면 학습 가능")


train 총 53,736건
라벨 분포: {0: np.int64(20604), 1: np.int64(9852), 2: np.int64(18480), 3: np.int64(4800)}
긴급(3) 비율: 8.93%  ← 5% 이상이면 학습 가능


In [5]:
import sys, os
os.chdir("/content/thisabled-ai")
for mod in list(sys.modules.keys()):
    if mod.startswith("src.") or mod == "src":
        del sys.modules[mod]
if "/content/thisabled-ai" not in sys.path:
    sys.path.insert(0, "/content/thisabled-ai")

from src.training.trainer import train_module1
result_final = train_module1("configs/module1_kcelectra_final.yaml", project_root=".")

import json
print("\n=== EXP-1 학습 완료 ===")
print(json.dumps(result_final["eval_metrics"], indent=2, ensure_ascii=False))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/504 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/450k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base-v2022
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the che

model.safetensors:   0%|          | 0.00/511M [00:00<?, ?B/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [6]:
%cd /content/thisabled-ai
!git pull


/content
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.20 KiB | 1.21 MiB/s, done.
From https://github.com/threeGuineas/thisabled-ai
   349966d..680c053  main       -> origin/main
Updating 349966d..680c053
Fast-forward
 src/training/trainer.py | 31 +++++++++++++++++++++----------
 1 file changed, 21 insertions(+), 10 deletions(-)


In [7]:
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("src.") or mod == "src":
        del sys.modules[mod]


In [8]:
from src.training.trainer import train_module1
result_final = train_module1("configs/module1_kcelectra_final.yaml", project_root=".")

import json
print("\n=== EXP-1 학습 완료 ===")
print(json.dumps(result_final["eval_metrics"], indent=2, ensure_ascii=False))


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base-v2022
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the che

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== EXP-1 학습 완료 ===
{
  "eval_loss": 0.6350276470184326,
  "eval_macro_f1": 0.7622952994991644,
  "eval_emergency_recall": 0.0,
  "eval_auc_pr": 0.8468932446483901,
  "eval_f1_class0": 0.8281573498964804,
  "eval_f1_class1": 0.6562756357670222,
  "eval_f1_class2": 0.8024529128339903,
  "eval_runtime": 3.4458,
  "eval_samples_per_second": 1717.139,
  "eval_steps_per_second": 26.989,
  "epoch": 3.0
}


In [9]:
import pandas as pd
df = pd.read_parquet("data/processed/train.parquet")
print(f"긴급(3) 비율: {(df['label']==3).mean()*100:.2f}%")


긴급(3) 비율: 8.93%


In [10]:
import pandas as pd, torch, numpy as np, json
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from src.training.dataset import RiskTextDataset
from src.evaluation.metrics import compute_classification_metrics

ckpt = result_final["checkpoint_dir"]
tok = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForSequenceClassification.from_pretrained(ckpt).cuda().eval()

def infer(p):
    ds = RiskTextDataset(p, tok, 128)
    loader = DataLoader(ds, batch_size=64, collate_fn=DataCollatorWithPadding(tok))
    lg, lb = [], []
    with torch.no_grad():
        for b in loader:
            lb.append(b.pop("labels").numpy())
            b = {k: v.cuda() for k, v in b.items()}
            lg.append(model(**b).logits.cpu().numpy())
    return np.concatenate(lg), np.concatenate(lb)

# 1) 시드 test (0/1/2 클래스 성능 변화 확인)
lg, lb = infer("data/processed/test.parquet")
m = compute_classification_metrics(lb, lg.argmax(-1), torch.softmax(torch.tensor(lg), -1).numpy())
print(f"\n[시드 test] Macro-F1: {m['macro_f1']:.4f}")
for c in [0, 1, 2]:
    pc = m["per_class"].get(c, m["per_class"].get(str(c), {}))
    print(f"  class {c} F1: {pc.get('f1', 0):.4f}")

# 데이터셋별 분리
test_df = pd.read_parquet("data/processed/test.parquet").reset_index(drop=True)
for src in ("unsmile", "kold"):
    mask = (test_df["source"] == src).values
    md = compute_classification_metrics(lb[mask], lg.argmax(-1)[mask], torch.softmax(torch.tensor(lg), -1).numpy()[mask])
    print(f"  [{src}] macro_f1={md['macro_f1']:.4f}")

# 2) 합성 hold-out — 긴급 Recall + boundary FPR
lg2, lb2 = infer("data/processed/synthetic_holdout.parquet")
pred2 = lg2.argmax(-1)
df_h = pd.read_parquet("data/processed/synthetic_holdout.parquet").reset_index(drop=True)
emerg = (df_h["label"] == 3).values
bound = (df_h["label"] < 3).values
print(f"\n[합성 hold-out]")
print(f"  긴급(3) Recall (n={emerg.sum()}): {(pred2[emerg] == 3).mean():.4f}  ← 목표 ≥ 0.75")
print(f"  Boundary FPR (n={bound.sum()}): {(pred2[bound] == 3).mean():.4f}")

# 메트릭 JSON 저장
import datetime
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")
out = {
    "model": "module1_final (CE α=[1,1.5,1,5], synth_repeat=8)",
    "seed_test": {
        "macro_f1": float(m["macro_f1"]),
        "per_class_f1": {c: float(m["per_class"].get(c, m["per_class"].get(str(c), {})).get("f1", 0)) for c in [0,1,2]},
    },
    "synthetic_holdout": {
        "emergency_recall": float((pred2[emerg] == 3).mean()),
        "boundary_fpr": float((pred2[bound] == 3).mean()),
        "n_emergency": int(emerg.sum()),
        "n_boundary": int(bound.sum()),
    },
}
from pathlib import Path
p = Path(f"reports/validation_reports/module1/final_{ts}.json")
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(json.dumps(out, indent=2, ensure_ascii=False))
print(f"\n✓ 메트릭 저장: {p}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


[시드 test] Macro-F1: 0.7643
  class 0 F1: 0.8339
  class 1 F1: 0.6571
  class 2 F1: 0.8020
  [unsmile] macro_f1=0.7941
  [kold] macro_f1=0.7355

[합성 hold-out]
  긴급(3) Recall (n=260): 1.0000  ← 목표 ≥ 0.75
  Boundary FPR (n=110): 0.0000

✓ 메트릭 저장: reports/validation_reports/module1/final_20260601_0522.json


In [11]:
!python scripts/train_lambdarank.py --mode embedding


=== [모듈 ②] LambdaMART (mode=embedding) ===
1. 프로필 1000건 생성 + user-level split
   train_users=800, test_users=200 (disjoint)
2. train/test 페어 별도 구성 (cold-start)
   train pairs=2400, test pairs=593
3. 피처 엔지니어링 (mode=embedding)
  - SBERT 텍스트 임베딩 추출 중 (mode=embedding)...
modules.json: 100% 229/229 [00:00<00:00, 1.00MB/s]
config_sentence_transformers.json: 100% 123/123 [00:00<00:00, 524kB/s]
README.md: 100% 4.86k/4.86k [00:00<00:00, 11.7MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 229kB/s]
config.json: 100% 744/744 [00:00<00:00, 5.12MB/s]
model.safetensors: 100% 442M/442M [00:03<00:00, 118MB/s]  
Loading weights: 100% 199/199 [00:00<00:00, 1047.66it/s, Materializing param=pooler.dense.weight]                              
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different tas

In [12]:
!python scripts/train_lambdarank.py --mode full


=== [모듈 ②] LambdaMART (mode=full) ===
1. 프로필 1000건 생성 + user-level split
   train_users=800, test_users=200 (disjoint)
2. train/test 페어 별도 구성 (cold-start)
   train pairs=2400, test pairs=593
3. 피처 엔지니어링 (mode=full)
  - SBERT 텍스트 임베딩 추출 중 (mode=full)...
Loading weights: 100% 199/199 [00:00<00:00, 910.77it/s, Materializing param=pooler.dense.weight]                               
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100% 38/38 [00:01<00:00, 29.48it/s]
Loading weights: 100% 199/199 [00:00<00:00, 1135.55it/s, Materializing param=pooler.dense.weight]                              
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+----------

In [13]:
%cd /content/thisabled-ai
!git pull


/content
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 10 (delta 4), reused 10 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 7.76 KiB | 3.88 MiB/s, done.
From https://github.com/threeGuineas/thisabled-ai
   680c053..6ee8c3b  main       -> origin/main
Updating 680c053..6ee8c3b
Fast-forward
 scripts/evaluate_fairness.py   | 231 ++++++++++++++---------------------------
 scripts/oversample_fairness.py | 145 ++++++++++++++++++++++++++
 src/evaluation/fairness.py     | 195 ++++++++++++++++++++++++++++++++++
 tests/test_fairness.py         |  71 +++++++++++++
 4 files changed, 489 insertions(+), 153 deletions(-)
 create mode 100644 scripts/oversample_fairness.py
 create mode 100644 src/evaluation/fairness.py
 create mode 100644 tests/test_fairness.py


In [14]:
!python scripts/evaluate_fairness.py \
    --model-dir {result_final["checkpoint_dir"]} \
    --output-json reports/validation_reports/module1/fairness_before.json


사용 장비: cuda
1. 모델 로딩: models/checkpoints/module1_final
Loading weights: 100% 201/201 [00:00<00:00, 1164.74it/s, Materializing param=electra.encoder.layer.11.output.dense.weight]             
2. Test 데이터 로딩: /content/thisabled-ai/data/processed/test.parquet
3. 예측 추출...
4. 보호집단별 공정성 평가...
Traceback (most recent call last):
  File "/content/thisabled-ai/scripts/evaluate_fairness.py", line 108, in <module>
    sys.exit(main())
             ^^^^^^
  File "/content/thisabled-ai/scripts/evaluate_fairness.py", line 101, in main
    print(f"\n✓ 결과 저장: {out_path.relative_to(ROOT)}")
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/pathlib.py", line 682, in relative_to
    raise ValueError(f"{str(self)!r} is not in the subpath of {str(other)!r}")
ValueError: 'reports/validation_reports/module1/fairness_before.json' is not in the subpath of '/content/thisabled-ai'


In [15]:
import json
data = json.loads(open("reports/validation_reports/module1/fairness_before.json").read())
print(json.dumps(data, indent=2, ensure_ascii=False))


{
  "unsmile_7_groups": {
    "groups": {
      "여성/가족": {
        "n": 203,
        "f1": 0.32142857142857145
      },
      "남성": {
        "n": 159,
        "f1": 0.3269230769230769
      },
      "성소수자": {
        "n": 143,
        "f1": 0.32
      },
      "인종/국적": {
        "n": 212,
        "f1": 0.3159636062861869
      },
      "연령": {
        "n": 81,
        "f1": 0.32489451476793246
      },
      "지역": {
        "n": 126,
        "f1": 0.49800796812749004
      },
      "종교": {
        "n": 146,
        "f1": 0.3202846975088968
      }
    },
    "max_gap": 0.18204436184130313,
    "n_groups_measured": 7
  },
  "kold_top_groups": {
    "groups": {
      "race-others": {
        "n": 155,
        "f1": 0.2737642585551331
      },
      "others-feminist": {
        "n": 138,
        "f1": 0.30446194225721784
      },
      "gender-LGBTQ+": {
        "n": 120,
        "f1": 0.2646566164154104
      },
      "religion-islam": {
        "n": 117,
        "f1": 0.305555555555555